In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Flatten, Dense, BatchNormalization

# 1. Load and preprocess (updated to not log-transform tp)
def load_and_prepare(path, suffix):
    df = pd.read_csv(path, parse_dates=['valid_time'])
    df = df.sort_values('valid_time')
    df['tp'] *= 1000  # Convert m to mm
    df = df[['valid_time', 'u10', 'v10', 't2m', 'sp', 'tp']]
    df.columns = ['valid_time'] + [f'{col}_{suffix}' for col in df.columns if col != 'valid_time']
    return df

# Load all 9 regions
df_center = load_and_prepare('data/brazil.csv', 'center')
df_north = load_and_prepare('data/brazil_north.csv', 'north')
df_south = load_and_prepare('data/brazil_south.csv', 'south')
df_east = load_and_prepare('data/brazil_east.csv', 'east')
df_west = load_and_prepare('data/brazil_west.csv', 'west')
df_ne = load_and_prepare('data/brazil_ne.csv', 'north-east')
df_nw = load_and_prepare('data/brazil_nw.csv', 'north-west')
df_se = load_and_prepare('data/brazil_se.csv', 'south-east')
df_sw = load_and_prepare('data/brazil_sw.csv', 'south-west')

# Merge on valid_time
df = df_center
for regional_df in [df_north, df_south, df_east, df_west, df_ne, df_nw, df_se, df_sw]:
    df = df.merge(regional_df, on='valid_time')

df = df.sort_values('valid_time').reset_index(drop=True)

# 2. Add cyclical time features
df['dayofyear'] = df['valid_time'].dt.dayofyear
df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

# 3. Define feature groups (including tp for all regions as features, but target is tp_center)
regions = ['center', 'north', 'south', 'east', 'west',
           'north-east', 'north-west', 'south-east', 'south-west']

all_meteorological_features = []
for region in regions:
    all_meteorological_features += [f'{var}_{region}' for var in ['u10', 'v10', 't2m', 'sp', 'tp']]

# Define the 3x3 grid layout (rows: NW, W, SW; columns: N, Center, S; wait, adjust to standard grid)
# Assuming layout:
# Row 0: north-west, north, north-east
# Row 1: west, center, east
# Row 2: south-west, south, south-east
grid_regions = [
    ['north-west', 'north', 'north-east'],
    ['west', 'center', 'east'],
    ['south-west', 'south', 'south-east']
]

# Features per region (excluding tp_center as target, but including others' tp)
features_per_region = ['u10', 'v10', 't2m', 'sp', 'tp']  # 5 features
num_features = len(features_per_region)

# Prepare target (tp_center, not log-transformed)
target = df['tp_center']

# Extract features for each region in grid order
features_grid = []
for row in grid_regions:
    row_features = []
    for region in row:
        region_features = [f'{var}_{region}' for var in features_per_region]
        row_features.append(df[region_features].values)
    features_grid.append(np.stack(row_features, axis=1))  # Shape: (time, 3 regions in row, num_features)

features_grid = np.stack(features_grid, axis=1)  # Shape: (time, 3, 3, num_features)

# Add cyclical time features (broadcast to grid)
time_features = df[['dayofyear_sin', 'dayofyear_cos']].values
time_features_expanded = np.repeat(time_features[:, np.newaxis, np.newaxis, :], 3, axis=1)
time_features_expanded = np.repeat(time_features_expanded, 3, axis=2)  # Shape: (time, 3, 3, 2)

# Combine meteorological and time features
full_features = np.concatenate([features_grid, time_features_expanded], axis=-1)  # Shape: (time, 3, 3, num_features + 2)

# Scale features (flatten temporarily for scaling)
scaler = StandardScaler()
time_steps, h, w, c = full_features.shape
full_features_flat = full_features.reshape(-1, c)
full_features_scaled_flat = scaler.fit_transform(full_features_flat)
full_features_scaled = full_features_scaled_flat.reshape(time_steps, h, w, c)

# Create sequences
seq_length = 24  # 24 days input
pred_length = 7  # Predict 7 days ahead
X, y = [], []
for i in range(len(full_features_scaled) - seq_length - pred_length + 1):
    X.append(full_features_scaled[i:i + seq_length])
    y.append(target.iloc[i + seq_length:i + seq_length + pred_length].values)

X = np.array(X)  # Shape: (num_samples, seq_length, 3, 3, c)
y = np.array(y)  # Shape: (num_samples, pred_length)

# Split into train/val/test (temporal split: 70/15/15)
train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))
test_size = len(X) - train_size - val_size

X_train, X_val, X_test = X[:train_size], X[train_size:train_size + val_size], X[train_size + val_size:]
y_train, y_val, y_test = y[:train_size], y[train_size:train_size + val_size], y[train_size + val_size:]

# Build ConvLSTM model
model = Sequential()
model.add(ConvLSTM2D(filters=32, kernel_size=(3, 3), input_shape=(seq_length, 3, 3, c), padding='same', return_sequences=False))
model.add(BatchNormalization())
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(pred_length))  # Output 7 predictions
model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), verbose=1)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate for each of the 7 days (on original mm scale, no inverse transform needed since no log)
for j in range(pred_length):
    r2 = r2_score(y_test[:, j], y_pred[:, j])
    mae = mean_absolute_error(y_test[:, j], y_pred[:, j])
    mse = mean_squared_error(y_test[:, j], y_pred[:, j])
    print(f"Day {j+1}: R2={r2:.4f}, MAE={mae:.4f}, MSE={mse:.4f}")
